# MIPDS Milestone 5 — Detection and Spatial Features
**Aaron Tobias · CSC 5352 Deep Learning**

**Status: experiment executed in Colab; written responses completed from its saved outputs.**

This notebook uses the actual Milestone 4 fine-tuned ResNet-50 for spatial-map inspection and representation comparison, and a separate COCO-pretrained YOLOv8-nano detector. YOLO does not consume the fine-tuned ResNet features in this implementation. The original participant-disjoint train/validation/test split is preserved; the two linear probes use 2,531 training frames and 559 validation frames, with no test frames used for this comparison.

This repository copy retains the numerical outputs and written reviews. The original reviewed export includes the heatmaps and detector overlays. The threshold summary was refreshed locally from the 60 saved per-image results after the written assessments were completed; model inference and probe fitting were not rerun during this editing pass. The original final runtime report is explicitly retained as a historical report, followed by a separate post-edit review.

This temporary module recognizes seven exercise types from public side-view recordings. It does not establish client identity, squat phase, movement quality, joint geometry, or completed repetitions. A final clean Colab run against the saved Drive project remains to be verified.


## Inputs and run order

1. Finish both training stages, the prediction audit, and the journal in the companion Milestone 4 notebook.
2. Keep the same project folder (`/content/mipds_public_exercises` by default) with its frames, `classification_manifest.csv`, and `milestone4_outputs/m4_finetuned.keras` plus `m4_run_record.json`. If you mount Drive, set the same `ROOT` in both notebooks.
3. Milestone 4 prepares the three manifests below. Review their content and complete the two empty scene rows in `heatmap_manifest.csv` with real images: multiple people, and a person with another recognizable object. Manually verify the single-person row too.
4. In Colab, select a GPU runtime, run cells in order, and fill every response and journal cell from the rendered outputs. Use the supplied course notebook and rubric as the reference; another starter is not required to use this prepared workflow.

| Manifest | Required columns | Meaning |
|---|---|---|
| `classification_manifest.csv` | `path,label,split,client_id,session_id` | Reuse real Milestone 4 images, inherited exercise labels, and group assignments. |
| `heatmap_manifest.csv` | `image_id,path,scene_type,scene_note` | Exactly three distinct images: `single`, `same_class_multiple`, and `mixed_classes`. |
| `detection_manifest.csv` | `image_id,path,scene_note` | Exactly 20 distinct domain images with notes describing visible content. |

For the detection domain, use the rubric's **indoor scenes** option: home/gym exercise environments. Include challenging views, clutter, lighting differences, or occlusion when actually available; selecting only successful detections weakens the comparison. Milestone 4 defaults to its 20 audit frames, which may have limited scene diversity; you can deliberately replace this detector sample with 20 varied domain images. The classification split must remain unchanged. Specialized gym equipment may not have a COCO label.

Use recordings and other people's images with permission for course submission, and avoid identifiable bystanders. The detector's COCO labels form a separate label space from the squat-position classifier. The current classifier uses held-out public participants, with scope limited to this dataset and side-view setup; performance on personal/client recordings remains unmeasured.


In [ ]:
# Dependency setup. Colab normally includes TensorFlow; use the Milestone 4
# package versions when needed for checkpoint compatibility.
import importlib.util
import subprocess
import sys

packages = ["ultralytics>=8.3,<9", "scikit-learn>=1.4,<2", "pandas", "pillow", "matplotlib"]
if importlib.util.find_spec("tensorflow") is None:
    packages.append("tensorflow>=2.16,<3")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])


0

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import math
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import tensorflow as tf
from IPython.display import display, Markdown
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from ultralytics import YOLO

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

versions = {name: importlib.metadata.version(name) for name in
            ["tensorflow", "ultralytics", "scikit-learn", "numpy", "pandas", "pillow"]}
display(pd.Series(versions, name="installed_version"))


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


tensorflow       2.21.0
ultralytics     8.4.175
scikit-learn      1.6.1
numpy             2.1.3
pandas            2.2.3
pillow           11.3.0
Name: installed_version, dtype: object

In [ ]:
# Same project folder as the companion Milestone 4 notebook.
# Set this to the folder containing your MIPDS experiment files.
PROJECT_DIR = "/content/mipds_public_exercises"
ROOT = Path(PROJECT_DIR).expanduser()
DRIVE_MOUNT = Path("/content/drive")
if ROOT.is_relative_to(DRIVE_MOUNT):
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT))
ROOT = ROOT.resolve()
M4_RECORD_PATH = ROOT / "milestone4_outputs" / "m4_run_record.json"
if not M4_RECORD_PATH.is_file():
    raise FileNotFoundError("Run the companion Milestone 4 notebook on the public exercise dataset first; keep its run record with the checkpoint.")
m4_record = json.loads(M4_RECORD_PATH.read_text(encoding="utf-8"))
required_record = {"milestone", "backbone", "nested_backbone", "feature_layer", "input_size",
                   "preprocessing", "preprocessing_location", "resize_method", "classes", "evaluation_scope",
                   "checkpoint_path", "checkpoint_sha256", "classification_manifest_sha256",
                   "frozen_weight_checks", "stages", "task", "dataset"}
if not required_record.issubset(m4_record):
    raise ValueError("Milestone 4 run record is incomplete; export it from the companion training notebook.")
if m4_record["milestone"] != 4 or len(m4_record["stages"]) != 2:
    raise ValueError("Require the completed two-stage Milestone 4 training record.")
if not all(m4_record["frozen_weight_checks"].get(key, False)
           for key in ["stage1", "stage2", "stage2_unfrozen_changed"]):
    raise ValueError("Milestone 4 weight-freezing checks did not pass.")
if m4_record["preprocessing_location"] != "outside_saved_model":
    raise ValueError("This companion adapter expects external preprocessing, as exported by Milestone 4.")

CHECKPOINT = ROOT / m4_record["checkpoint_path"]
CLASSIFICATION_CSV = ROOT / "classification_manifest.csv"
HEATMAP_CSV = ROOT / "heatmap_manifest.csv"
DETECTION_CSV = ROOT / "detection_manifest.csv"
OUTPUT = ROOT / "milestone5_outputs"
BACKBONE_FAMILY = m4_record["backbone"]
NESTED_BACKBONE_NAME = m4_record["nested_backbone"]
FEATURE_LAYER = m4_record["feature_layer"]
PREPROCESSING = m4_record["preprocessing"]
INPUT_SIZE = tuple(m4_record["input_size"])
RESIZE_METHOD = m4_record["resize_method"]
EVALUATION_SCOPE = m4_record["evaluation_scope"]
M4_CLASSES = m4_record["classes"]
M4_TASK = m4_record["task"]
M4_DATASET = m4_record["dataset"]
M4_SETTINGS_VERIFIED = False  # Becomes True after checkpoint/data hashes are verified below.
CUSTOM_OBJECTS = {}

BATCH_SIZE = 8
THRESHOLDS = (0.25, 0.50, 0.75)
YOLO_WEIGHTS = "yolov8n.pt"
YOLO_IMGSZ = 640
YOLO_IOU = 0.70
YOLO_DEVICE = "cpu"  # Set 0 to use the GPU; CPU reduces GPU memory competition.
PROBE_C = 1.0
PROBE_CLASS_WEIGHT = "balanced"
PROBE_MAX_ITER = 2000
MAX_FEATURE_MEMORY_MIB = 2048
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Using recorded M4 settings:", BACKBONE_FAMILY, PREPROCESSING, INPUT_SIZE, EVALUATION_SCOPE)


Mounted at /content/drive
Using recorded M4 settings: resnet50 resnet50 (224, 224) unseen_people


## Check the data and preserve the evaluation split

The classifier manifest must reuse Milestone 4's exact saved file, verified by its SHA-256 hash. Recording sessions, source videos, and identical image bytes cannot cross splits. For `unseen_people`, clients must also be disjoint. For `within_person_sessions`, the same person can appear in different sessions and splits, with conclusions limited to that person; pretending their sessions are different clients would give misleading evidence.

These checks cannot detect all near duplicates or incorrectly entered session IDs. Verify the actual recording boundaries. The approximate 70/15/15 design is not a reason to re-split the experiment here. Any real classification-data correction requires a new Milestone 4 run and matching record.

Missing detector/scene manifests get empty headers only. Complete them with real content; no replacement images or labels are generated. Milestone 4's defaults preserve existing detector and scene manifests on subsequent runs.


In [ ]:
schemas = {
    CLASSIFICATION_CSV: ["path", "label", "split", "client_id", "session_id"],
    HEATMAP_CSV: ["image_id", "path", "scene_type", "scene_note"],
    DETECTION_CSV: ["image_id", "path", "scene_note"],
}
for path, columns in schemas.items():
    if not path.exists():
        pd.DataFrame(columns=columns).to_csv(path, index=False)
        print(f"Created empty manifest: {path.name}; fill it with real data.")

def resolve_image_path(value):
    path = Path(value)
    return (ROOT / path).resolve() if not path.is_absolute() else path.resolve()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def read_manifest(path, required):
    table = pd.read_csv(path, dtype=str, keep_default_na=False)
    missing = set(required) - set(table.columns)
    if missing:
        raise ValueError(f"{path.name}: missing columns {sorted(missing)}")
    if table.empty:
        raise ValueError(f"{path.name} is empty. Add your real images and metadata.")
    if table[required].apply(lambda col: col.str.strip().eq("")).any().any():
        raise ValueError(f"{path.name}: required fields cannot be blank.")
    table["resolved_path"] = table["path"].map(resolve_image_path)
    absent = [str(p) for p in table.resolved_path if not p.is_file()]
    if absent:
        raise FileNotFoundError(f"Missing image files (first three): {absent[:3]}")
    if table.resolved_path.duplicated().any():
        raise ValueError(f"{path.name}: image paths must be distinct.")
    table["sha256"] = table.resolved_path.map(sha256_file)
    return table

def validate_classification(table):
    splits = set(table.split)
    if not {"train", "validation"}.issubset(splits) or not splits.issubset({"train", "validation", "test"}):
        raise ValueError("Use train, validation, and optional test; keep the actual Milestone 4 assignments.")
    keys = ["session_id", "sha256"]
    if "video_path" in table.columns: keys.append("video_path")
    if EVALUATION_SCOPE == "unseen_people": keys.append("client_id")
    elif EVALUATION_SCOPE != "within_person_sessions": raise ValueError("Unknown M4 evaluation scope.")
    for key in keys:
        if (table.groupby(key)["split"].nunique() > 1).any():
            raise ValueError(f"Leakage: {key} crosses splits. Repair and rerun Milestone 4.")
    if "frame_id" in table.columns and table.frame_id.duplicated().any():
        raise ValueError("Classification frame IDs must be distinct.")
    synthetic_labels = {"retriever", "cat", "costume_dog", "scan_style"}
    if set(table.label).issubset(synthetic_labels):
        raise ValueError("The separate synthetic Chapter 4 exercise is not MIPDS milestone evidence.")
    train_labels = set(table.loc[table.split.eq("train"), "label"])
    val_labels = set(table.loc[table.split.eq("validation"), "label"])
    if len(train_labels) < 2 or not val_labels.issubset(train_labels):
        raise ValueError("Need at least two training classes and training coverage of every validation label.")

if sha256_file(CLASSIFICATION_CSV) != m4_record["classification_manifest_sha256"]:
    raise ValueError("Classification manifest differs from the one used to train Milestone 4.")
classification = read_manifest(CLASSIFICATION_CSV, schemas[CLASSIFICATION_CSV])
validate_classification(classification)
if set(classification.label) != set(M4_CLASSES):
    raise ValueError("Classification labels differ from Milestone 4's recorded label order.")
heatmap_images = read_manifest(HEATMAP_CSV, schemas[HEATMAP_CSV])
detector_images = read_manifest(DETECTION_CSV, schemas[DETECTION_CSV])

# Correct scene notes after visual inspection; leave source files unchanged.
heatmap_images.loc[heatmap_images.image_id.eq("single_person"), "scene_note"] = "One dominant centered exerciser; background people also visible."
heatmap_images.loc[heatmap_images.image_id.eq("person_and_object"), "scene_note"] = "Push-up exerciser beside floor dumbbells and a background exercise bench."

if len(heatmap_images) != 3 or set(heatmap_images.scene_type) != {"single", "same_class_multiple", "mixed_classes"}:
    raise ValueError("Stage 1 needs exactly one image for each of the three scene types.")
for name, table in [("Stage 1", heatmap_images), ("Stage 2", detector_images)]:
    if table.image_id.duplicated().any() or table.sha256.duplicated().any():
        raise ValueError(f"{name}: use distinct IDs and distinct image content.")
if len(detector_images) != 20:
    raise ValueError(f"Stage 2 requires exactly 20 images; found {len(detector_images)}.")

display(pd.crosstab(classification.label, classification.split))
display(detector_images[["image_id", "path", "scene_note"]])
print("No test images will be used to fit or evaluate the Stage 3 probes.")


split               test  train  validation
label                                      
bent_over_row         72    396          96
bicep_curl            72    336          72
dumbbell_side_bend    96    406          96
push_up               60    357          81
shoulder_press        90    348          48
squat                 84    324          94
triceps_extension     96    364          72

                     image_id  \
0   b7a93b6e5688c108_00000075   
1   8e1d58ae1ba4bd9c_00000105   
2   66e3e16f1bed521d_00000135   
3   3c2d66e9f8ae3960_00000265   
4   e565a4acf607d637_00000347   
5   0ffd0f439a63e0b7_00000046   
6   61624cea71c762cc_00000524   
7   0eb31af6c0e5d26d_00000135   
8   5bac5f42a97e3766_00000154   
9   e565a4acf607d637_00000272   
10  c977a4f4a4de27d9_00000044   
11  548350fd5ac497b8_00000185   
12  7259035d2ffaf7e3_00000270   
13  f379c74999a7bd7c_00000300   
14  67e3fef9fbf3b516_00000188   
15  e4142b3b5f20ee63_00000270   
16  137ed68256be045c_00000361   
17  a5441cc93036766f_00000018   
18  fa3a53594382a7e7_00000253   
19  b7a93b6e5688c108_00000405   

                                                 path  \
0   frames/b7a93b6e5688c108/b7a93b6e5688c108_00000...   
1   frames/8e1d58ae1ba4bd9c/8e1d58ae1ba4bd9c_00000...   
2   frames/66e3e16f1bed521d/66e3e16f1bed521d_00000...   
3   frames/3c2d66e9f8ae3960/3c2d66e9f8ae3960_00000...   
4   frames/e565a4acf6

No test images will be used to fit or evaluate the Stage 3 probes.


## Load the real fine-tuned spatial extractor

The companion Milestone 4 model contains a nested ResNet-50. Its recorded `conv5_block3_out` layer supplies spatial maps before global averaging; with 224×224 input they are normally 7×7×2048. The code verifies the actual tensor. Keras ResNet preprocessing remains outside the model, exactly as in Milestone 4's validation pipeline.

The checkpoint hash and classification-manifest hash must match the completed training record. Load without compiling because this notebook uses a frozen extractor and new linear probes rather than resuming the optimizer. The model's BatchNormalization is kept in inference mode during extraction.


In [ ]:
if not CHECKPOINT.is_file():
    raise FileNotFoundError("Keep the actual fine-tuned Milestone 4 checkpoint in the shared project folder.")
if sha256_file(CHECKPOINT) != m4_record["checkpoint_sha256"]:
    raise ValueError("Checkpoint differs from the model exported in the Milestone 4 run record.")
saved_model = tf.keras.models.load_model(CHECKPOINT, compile=False, custom_objects=CUSTOM_OBJECTS)
M4_SETTINGS_VERIFIED = True
saved_model.summary()
print("Top-level layer names:", [layer.name for layer in saved_model.layers])


Model: "mipds_exercise_recognition"


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ preprocessed_image (InputLayer) │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pool             │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ head_dropout (Dropout)          │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ exercise_class (Dense)          │ (None, 7)              │        14,343 │

 Total params: 23,602,055 (90.03 MB)


 Trainable params: 22,055,943 (84.14 MB)


 Non-trainable params: 1,546,112 (5.90 MB)


Top-level layer names: ['preprocessed_image', 'resnet50', 'global_average_pool', 'head_dropout', 'exercise_class']


In [ ]:
if not M4_SETTINGS_VERIFIED or PREPROCESSING not in {"resnet50", "raw_0_255", "raw_0_1", "minus1_1"}:
    raise ValueError("Run the checkpoint and manifest verification cells; use the recorded Milestone 4 preprocessing.")
if BACKBONE_FAMILY not in {"resnet50", "efficientnetb0"}:
    raise ValueError("Use your actual ResNet-50 or EfficientNet-B0 backbone.")
feature_source = saved_model.get_layer(NESTED_BACKBONE_NAME) if NESTED_BACKBONE_NAME else saved_model
if not isinstance(feature_source, tf.keras.Model):
    raise TypeError("NESTED_BACKBONE_NAME must identify the trained backbone model.")
spatial_layer = feature_source.get_layer(FEATURE_LAYER)
spatial_model = tf.keras.Model(feature_source.inputs, spatial_layer.output)
spatial_model.trainable = False
if len(spatial_model.inputs) != 1 or len(spatial_model.output_shape) != 4:
    raise ValueError("Expected one input and a channel-last spatial feature output, before pooling.")
expected_channels = {"resnet50": 2048, "efficientnetb0": 1280}[BACKBONE_FAMILY]
if spatial_model.output_shape[-1] != expected_channels:
    raise ValueError("Wrong feature layer or tensor layout: check the last convolutional block.")
shape = spatial_model.input_shape
if len(shape) != 4 or shape[-1] != 3:
    raise ValueError("This adapter requires channel-last RGB input.")
for actual, configured in zip(shape[1:3], INPUT_SIZE):
    if actual is not None and actual != configured:
        raise ValueError("INPUT_SIZE does not match the saved extractor.")

def load_rgb(path):
    with Image.open(path) as image:
        return np.asarray(ImageOps.exif_transpose(image).convert("RGB"))

def read_model_input(path):
    # Match the original validation resize, color handling, and normalization.
    # This implementation stretches to INPUT_SIZE without crop or letterbox.
    # Adapt this AND the heatmap alignment if Milestone 4 used different geometry.
    return tf.image.resize(load_rgb(path), INPUT_SIZE, method=RESIZE_METHOD).numpy().astype(np.float32)

def preprocess_batch(batch):
    batch = np.array(batch, dtype=np.float32, copy=True)
    if PREPROCESSING == "resnet50":
        return tf.keras.applications.resnet50.preprocess_input(batch)
    if PREPROCESSING == "raw_0_1":
        return batch / 255.0
    if PREPROCESSING == "minus1_1":
        return batch / 127.5 - 1.0
    return batch

def extract_maps(paths):
    batch = preprocess_batch(np.stack([read_model_input(p) for p in paths]))
    maps = np.asarray(spatial_model(batch, training=False), dtype=np.float32)
    if maps.ndim != 4 or not np.isfinite(maps).all():
        raise ValueError("Expected finite (batch, height, width, channels) maps.")
    if min(maps.shape[1:3]) < 4:
        raise ValueError("Spatial map must support the requested 4×4 pooling grid.")
    return maps

probe_map = extract_maps([heatmap_images.resolved_path.iloc[0]])
print("Verified feature tensor:", probe_map.shape)
print("Checkpoint SHA-256:", sha256_file(CHECKPOINT))


/usr/local/lib/python3.13/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input_layer_1']
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)


Verified feature tensor: (1, 7, 7, 2048)
Checkpoint SHA-256: 0d59d2a0080d4fda25d1fa02742f1b98d976aa1578cf49b535055797d3f91152


## Stage 1 — Spatial heatmaps for three scenes

Average each map over its channels and overlay it on the original image. These are **channel-mean activation maps**, not class-specific Grad-CAM explanations, causal evidence, or a measurement of an attention mechanism. Per-image min/max normalization makes spatial patterns visible, so color intensity should not be compared as an absolute activation score across different images. A flat activation map is displayed as flat, not as invented focus.


In [ ]:
def channel_mean_heatmap(feature_map):
    values = np.asarray(feature_map, dtype=np.float32).mean(axis=-1)
    low, high = float(values.min()), float(values.max())
    return np.zeros_like(values) if high - low <= 1e-8 else (values - low) / (high - low)

heatmap_files = []
scene_order = ["single", "same_class_multiple", "mixed_classes"]
for index, scene_type in enumerate(scene_order, start=1):
    row = heatmap_images.loc[heatmap_images.scene_type.eq(scene_type)].iloc[0]
    rgb = load_rgb(row.resolved_path)
    feature_map = extract_maps([row.resolved_path])[0]
    heat = channel_mean_heatmap(feature_map)
    aligned = np.asarray(Image.fromarray(heat, mode="F").resize(
        (rgb.shape[1], rgb.shape[0]), resample=Image.Resampling.BILINEAR))
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].imshow(rgb)
    axes[0].set_title("Original")
    axes[1].imshow(heat, cmap="inferno", vmin=0, vmax=1)
    axes[1].set_title(f"Mean activation · {feature_map.shape}")
    axes[2].imshow(rgb)
    axes[2].imshow(aligned, cmap="inferno", alpha=0.45, vmin=0, vmax=1)
    axes[2].set_title("Activation overlay")
    for ax in axes:
        ax.axis("off")
    fig.suptitle(f"Image {index} · {row.image_id} · {row.scene_note}")
    fig.tight_layout()
    figure_path = OUTPUT / f"stage1_image_{index}.png"
    fig.savefig(figure_path, dpi=160, bbox_inches="tight")
    heatmap_files.append(figure_path)
    plt.show()
    plt.close(fig)


/tmp/ipykernel_2749/1985571472.py:13: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  aligned = np.asarray(Image.fromarray(heat, mode="F").resize(


[Public copy: 3 figure output(s) with dataset frames omitted.]


### Stage 1 response — Image 1: single centered object
In `b7a93b6e5688c108_00000075`, the strongest activation lies around the foreground exerciser's waist, hips, upper legs, and nearby hand/weight region, while the head, ceiling, and most of the floor receive less activation. This is consistent with a representation responding to exercise-related body and setup cues, but the coarse map cannot separate clothing, anatomy, equipment, and adjacent background. The image has one dominant centered subject, with additional people visible in the background; it is not a scene containing exactly one person. The visualization is a normalized channel-mean activation map from a 7×7×2,048 tensor, not class-specific GradCAM or proof that a particular region caused the prediction.


### Stage 1 response — Image 2: multiple instances of one class
In `11838f42fa861e23_00000107`, the orange-shirted foreground exerciser dominates the activation, particularly around the hips, hands, dumbbells, and upper legs. The people farther left and behind the main subject have substantially weaker activation, so the response does not spread equally across everyone visible. Global average pooling merges each channel's responses across all spatial locations into one value and loses which person or location produced them. This heatmap therefore suggests foreground emphasis, but it neither counts people nor establishes that the backbone independently detected every person.


### Stage 1 response — Image 3: complex scene with different classes
In `0ac8ad70d6ba1d75_00000179`, a person performs a push-up, with dumbbells on the floor and an exercise bench in the upper-right background. The saved figure's original title says “Person holding a visible dumbbell”; that description is incorrect, and the person is not holding one in this frame. The strongest activation is near the hips and upper legs, with an extended response toward the lower-body/equipment area on the right rather than the visually prominent head and hands. This is compatible with exercise-domain adaptation, but one channel-mean map cannot show whether the response comes from posture, clothing, floor texture, or nearby equipment. The person, bench, and dumbbells make this a scene with different object types for spatial inspection; dumbbell is absent from the detector's exported COCO class list, and this heatmap does not demonstrate separate detection of any object.


## Stage 2 — YOLOv8-nano threshold experiment

Run all **20 images independently at each threshold**: 60 image/threshold records. Keep resolution, NMS IoU, and class handling fixed. Include zero-detection images in each average; their highest confidence is undefined, recorded as an empty value rather than zero. Record all detected COCO classes and also the number of `person` boxes because people are central to this application.

For qualitative assessment, inspect the rendered boxes for all 20 images at all three thresholds. Confidence alone does not establish correctness. No precision, recall, mAP, or probability calibration is claimed without appropriate ground-truth annotation/evaluation. Threshold choice needs the actual observed false boxes and missed people, especially occluded or distant clients.


In [ ]:
detector = YOLO(YOLO_WEIGHTS)
if detector.task != "detect" or len(detector.names) != 80 or "person" not in detector.names.values():
    raise ValueError("Use the required COCO-pretrained YOLOv8-nano detection checkpoint.")
print("YOLO classes:", detector.names)
if Path(YOLO_WEIGHTS).is_file():
    print("YOLO checkpoint SHA-256:", sha256_file(YOLO_WEIGHTS))

detector_records = []
boxes_payload = []
rendered_detections = {}
for threshold in THRESHOLDS:
    rendered_detections[threshold] = []
    for image_number, row in enumerate(detector_images.itertuples(index=False), start=1):
        result = detector.predict(
            source=load_rgb(row.resolved_path)[..., ::-1].copy(), conf=threshold, iou=YOLO_IOU,
            imgsz=YOLO_IMGSZ, device=YOLO_DEVICE, max_det=300,
            augment=False, agnostic_nms=False, verbose=False,
        )[0]
        boxes = result.boxes
        if boxes is None:
            confidences, class_ids, coordinates = np.array([]), np.array([], dtype=int), np.empty((0, 4))
        else:
            confidences = boxes.conf.cpu().numpy()
            class_ids = boxes.cls.cpu().numpy().astype(int)
            coordinates = boxes.xyxy.cpu().numpy()
        labels = [result.names[int(c)] for c in class_ids]
        detector_records.append({
            "image_id": row.image_id, "image_path": row.path,
            "scene_note": row.scene_note, "threshold": threshold,
            "detections": len(labels), "class_labels": ", ".join(labels) if labels else "none",
            "highest_confidence": float(confidences.max()) if len(confidences) else np.nan,
            "person_detections": labels.count("person"),
        })
        boxes_payload.append({
            "image_id": row.image_id, "threshold": threshold,
            "coordinate_system": "xyxy_pixels_in_exif_oriented_original_image",
            "image_height": int(result.orig_shape[0]), "image_width": int(result.orig_shape[1]),
            "source_model": "COCO-pretrained YOLOv8-nano; separate from M4 backbone",
            "boxes": [{"label": label, "confidence": float(conf), "xyxy": coord.tolist()}
                      for label, conf, coord in zip(labels, confidences, coordinates)],
        })
        # Ultralytics plotting returns BGR; convert to RGB for Matplotlib/Pillow.
        rendered = result.plot()[..., ::-1]
        rendered_path = OUTPUT / f"stage2_{threshold:.2f}_image_{image_number:02d}.jpg"
        Image.fromarray(rendered).save(rendered_path)
        rendered_detections[threshold].append((row.image_id, rendered_path))

detections = pd.DataFrame(detector_records)
assert len(detections) == 60
assert detections.groupby("threshold").size().eq(20).all()
detections.to_csv(OUTPUT / "stage2_per_image_detections.csv", index=False)
(OUTPUT / "stage2_boxes.json").write_text(json.dumps(boxes_payload, indent=2), encoding="utf-8")
for threshold in THRESHOLDS:
    display(Markdown(f"### Per-image results at confidence {threshold:.2f}"))
    display(detections.loc[detections.threshold.eq(threshold),
        ["image_id", "detections", "class_labels", "highest_confidence", "person_detections"]].reset_index(drop=True))


YOLO classes: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse', 65: 'remote', 66: 'keyboard', 67: 'cell phone

### Per-image results at confidence 0.25

                     image_id  detections  \
0   b7a93b6e5688c108_00000075           5   
1   8e1d58ae1ba4bd9c_00000105           5   
2   66e3e16f1bed521d_00000135           2   
3   3c2d66e9f8ae3960_00000265           6   
4   e565a4acf607d637_00000347           6   
5   0ffd0f439a63e0b7_00000046           6   
6   61624cea71c762cc_00000524           3   
7   0eb31af6c0e5d26d_00000135           5   
8   5bac5f42a97e3766_00000154           1   
9   e565a4acf607d637_00000272           5   
10  c977a4f4a4de27d9_00000044           5   
11  548350fd5ac497b8_00000185           5   
12  7259035d2ffaf7e3_00000270           5   
13  f379c74999a7bd7c_00000300           7   
14  67e3fef9fbf3b516_00000188           8   
15  e4142b3b5f20ee63_00000270           4   
16  137ed68256be045c_00000361           8   
17  a5441cc93036766f_00000018           7   
18  fa3a53594382a7e7_00000253           6   
19  b7a93b6e5688c108_00000405           7   

                                         class_labels 

### Per-image results at confidence 0.50

                     image_id  detections  \
0   b7a93b6e5688c108_00000075           4   
1   8e1d58ae1ba4bd9c_00000105           3   
2   66e3e16f1bed521d_00000135           2   
3   3c2d66e9f8ae3960_00000265           3   
4   e565a4acf607d637_00000347           4   
5   0ffd0f439a63e0b7_00000046           2   
6   61624cea71c762cc_00000524           2   
7   0eb31af6c0e5d26d_00000135           4   
8   5bac5f42a97e3766_00000154           1   
9   e565a4acf607d637_00000272           4   
10  c977a4f4a4de27d9_00000044           2   
11  548350fd5ac497b8_00000185           3   
12  7259035d2ffaf7e3_00000270           4   
13  f379c74999a7bd7c_00000300           4   
14  67e3fef9fbf3b516_00000188           5   
15  e4142b3b5f20ee63_00000270           4   
16  137ed68256be045c_00000361           5   
17  a5441cc93036766f_00000018           2   
18  fa3a53594382a7e7_00000253           3   
19  b7a93b6e5688c108_00000405           3   

                              class_labels  highest_co

### Per-image results at confidence 0.75

                     image_id  detections            class_labels  \
0   b7a93b6e5688c108_00000075           2          person, person   
1   8e1d58ae1ba4bd9c_00000105           1                  person   
2   66e3e16f1bed521d_00000135           2          person, person   
3   3c2d66e9f8ae3960_00000265           0                    none   
4   e565a4acf607d637_00000347           1                  person   
5   0ffd0f439a63e0b7_00000046           2          person, person   
6   61624cea71c762cc_00000524           0                    none   
7   0eb31af6c0e5d26d_00000135           2          person, person   
8   5bac5f42a97e3766_00000154           0                    none   
9   e565a4acf607d637_00000272           2          person, person   
10  c977a4f4a4de27d9_00000044           0                    none   
11  548350fd5ac497b8_00000185           1                  person   
12  7259035d2ffaf7e3_00000270           0                    none   
13  f379c74999a7bd7c_00000300     

In [ ]:
# Inspect ALL 20 images for EACH threshold. Four-image pages remain readable.
for threshold in THRESHOLDS:
    images = rendered_detections[threshold]
    for start in range(0, len(images), 4):
        fig, axes = plt.subplots(2, 2, figsize=(12, 9))
        for ax, (image_id, path) in zip(axes.flat, images[start:start + 4]):
            ax.imshow(load_rgb(path))
            count = int(detections.loc[(detections.threshold.eq(threshold)) &
                        (detections.image_id.eq(image_id)), "detections"].iloc[0])
            ax.set_title(f"{image_id} · threshold {threshold:.2f} · {count} boxes")
            ax.axis("off")
        fig.tight_layout()
        plt.show()
        plt.close(fig)


[Public copy: 15 figure output(s) with dataset frames omitted.]


In [ ]:
# Single-sentence assessments grounded in the saved 20-image visual review.
QUALITATIVE_ASSESSMENT = {0.25: 'The 0.25 threshold retains more background people but adds misleading equipment labels, including sports ball on a dumbbell in 8e1d58ae1ba4bd9c_00000105 and suitcase/cell phone on gym equipment in a5441cc93036766f_00000018.', 0.5: 'The 0.50 threshold removes many distracting low-score boxes while retaining useful foreground-person boxes in several scenes, but still misses the push-up exerciser in 3c2d66e9f8ae3960_00000265 and retains a false horse label in 67e3fef9fbf3b516_00000188.', 0.75: 'The 0.75 threshold gives sparse person-only outputs but leaves five images with no detections and misses the foreground exerciser in 8e1d58ae1ba4bd9c_00000105 while retaining a background person.'}

def summarize_thresholds(records, qualitative):
    summary = records.groupby("threshold", sort=True).agg(
        images=("image_id", "nunique"),
        average_detections_per_image=("detections", "mean"),
        average_person_detections_per_image=("person_detections", "mean"),
        zero_detection_images=("detections", lambda values: int(values.eq(0).sum())),
    ).reset_index()
    summary["qualitative_assessment"] = summary.threshold.map(
        lambda threshold: qualitative.get(threshold, "").strip() or "Assessment missing"
    )
    return summary

threshold_summary = summarize_thresholds(detections, QUALITATIVE_ASSESSMENT)
display(threshold_summary.rename(columns={
    "threshold": "Threshold", "average_detections_per_image": "Avg. detections per image",
    "average_person_detections_per_image": "Avg. person boxes per image",
    "zero_detection_images": "Images with no detections", "qualitative_assessment": "Qualitative assessment",
}).round(3))
threshold_summary.to_csv(OUTPUT / "stage2_threshold_summary.csv", index=False)


 Threshold  images  Avg. detections per image  Avg. person boxes per image  Images with no detections                                                                                                                                                                                                                                          Qualitative assessment
      0.25      20                       5.30                         4.45                          0                                 The 0.25 threshold retains more background people but adds misleading equipment labels, including sports ball on a dumbbell in 8e1d58ae1ba4bd9c_00000105 and suitcase/cell phone on gym equipment in a5441cc93036766f_00000018.
      0.50      20                       3.20                         2.90                          0 The 0.50 threshold removes many distracting low-score boxes while retaining useful foreground-person boxes in several scenes, but still misses the push-up exerciser in 3c2d66e9f8ae39

### Stage 2 review notes

All 20 images were reviewed at each of the three thresholds. At 0.25, the average was 5.30 boxes per image, including 4.45 person boxes; at 0.50, it was 3.20 boxes and 2.90 person boxes; at 0.75, it was 1.25 boxes, all labeled person. Zero-detection images increased from zero at 0.25 and 0.50 to five at 0.75. These are counts of model outputs, not measured detection precision, recall, or mAP, because ground-truth boxes were not supplied.

| Image ID | Direct observation from the saved figures | Review implication |
| --- | --- | --- |
| `8e1d58ae1ba4bd9c_00000105` | At 0.25, a dumbbell is labeled sports ball (about 0.49); at 0.50 that label disappears and the foreground rower remains boxed (about 0.73); at 0.75 only a background person remains. | A higher threshold can retain an unrelated person while losing the client of interest. |
| `a5441cc93036766f_00000018` | At 0.25, weights/equipment receive suitcase and cell-phone labels; at 0.50 those equipment boxes disappear. | The lower threshold adds distracting labels outside the exercise domain. |
| `3c2d66e9f8ae3960_00000265` | A person box surrounds the push-up exerciser at about 0.30 in the 0.25 result; 0.50 drops that box while retaining background people; 0.75 produces no boxes. | An image with detections can still lack the foreground exerciser. |
| `67e3fef9fbf3b516_00000188` | The foreground push-up exerciser is labeled horse at about 0.55 at both 0.25 and 0.50; 0.75 removes it and leaves background people. | Threshold changes cannot repair a confident class mismatch. |
| `137ed68256be045c_00000361` | At 0.75, background people remain boxed but the foreground exerciser disappears. | A nonempty detection result is insufficient for selecting the intended client. |

Some people appear in mirrors, so a person-box count is not a reliable count of distinct physical people. My provisional default is 0.50 for trainer-reviewed overlays, with an explicit missing-client check and an option to inspect the 0.25 result. No threshold in this experiment establishes autonomous person selection or reliable equipment recognition.


## Stage 3 — Global versus multi-scale spatial pooling

Global average pooling produces `C` features. Pooling at 1×1, 2×2, and 4×4 grids produces `(1 + 4 + 16) × C = 21C` features: **43,008** for 2048-channel ResNet-50 or **26,880** for 1280-channel EfficientNet-B0. It summarizes coarse location rather than providing pixel-level segmentation, object identity, or precise box coordinates.

A 7×7 map does not divide evenly into a 4×4 grid. This scaffold uses adaptive average-pooling bin boundaries `floor(i×H/g)` to `ceil((i+1)×H/g)`, and the equivalent boundaries in width. Adjacent bins can overlap, following adaptive pooling semantics; no border cells are discarded. Flatten each grid in row/column/channel order and concatenate scales in the order 1, 2, 4. Replace this helper with the instructor-provided function if its bin convention differs and describe the convention actually used.


In [ ]:
def global_average_pool(maps):
    maps = np.asarray(maps, dtype=np.float32)
    if maps.ndim != 4 or not np.isfinite(maps).all():
        raise ValueError("Expected finite NHWC feature maps.")
    return maps.mean(axis=(1, 2), dtype=np.float32)

def adaptive_average_grid(maps, grid):
    maps = np.asarray(maps, dtype=np.float32)
    if maps.ndim != 4 or not np.isfinite(maps).all():
        raise ValueError("Expected finite NHWC feature maps.")
    n, height, width, channels = maps.shape
    if not isinstance(grid, int) or grid < 1 or min(height, width) < grid:
        raise ValueError("Grid must be a positive integer supported by the spatial map.")
    pooled = np.empty((n, grid, grid, channels), dtype=np.float32)
    for row in range(grid):
        top = math.floor(row * height / grid)
        bottom = math.ceil((row + 1) * height / grid)
        for col in range(grid):
            left = math.floor(col * width / grid)
            right = math.ceil((col + 1) * width / grid)
            pooled[:, row, col, :] = maps[:, top:bottom, left:right, :].mean(
                axis=(1, 2), dtype=np.float32)
    return pooled

def multi_scale_spatial_pool(maps, grids=(1, 2, 4)):
    maps = np.asarray(maps, dtype=np.float32)
    return np.concatenate([
        adaptive_average_grid(maps, grid).reshape(maps.shape[0], -1)
        for grid in grids
    ], axis=1)

gap_example = global_average_pool(probe_map)
spatial_example = multi_scale_spatial_pool(probe_map)
assert spatial_example.shape[1] == 21 * probe_map.shape[-1]
np.testing.assert_allclose(spatial_example[:, :probe_map.shape[-1]], gap_example)
print("Actual GAP dimension:", gap_example.shape[1])
print("Actual multi-scale dimension:", spatial_example.shape[1])
print("The first 1×1 block matches GAP.")


Actual GAP dimension: 2048
Actual multi-scale dimension: 43008
The first 1×1 block matches GAP.


### Controlled linear-probe comparison

Freeze the **same trained backbone** and extract both representations from each identical input batch. Fit a separate standard scaler using **training features only**, followed by logistic regression with the same predeclared `C`, class weighting, iteration cap, and solver. Fit on the existing training split and evaluate the existing validation split once. The new probes are a representation comparison, not an exact reproduction of the Milestone 4 classifier's score or an additional fine-tuning stage.

The larger vector increases memory and classifier parameter count and may overfit a small dataset. Overall validation accuracy is required by the rubric; macro-F1 and balanced accuracy expose some effects of class imbalance. With grouped video data, image count is not the same as the number of independent clients/sessions. Neither this small comparison nor the 20-image detection exercise establishes deployment safety or guarantees 30 fps.


In [ ]:
train_rows = classification.loc[classification.split.eq("train")].reset_index(drop=True)
val_rows = classification.loc[classification.split.eq("validation")].reset_index(drop=True)
projected_bytes = (len(train_rows) + len(val_rows)) * expected_channels * 22 * 4
print(f"Pooled feature storage estimate: {projected_bytes / 1024**2:.1f} MiB (before scaler/solver copies).")
if projected_bytes / 1024**2 > MAX_FEATURE_MEMORY_MIB:
    raise MemoryError("Feature matrices exceed the configured budget; use a suitable runtime or memmap, preserving the split.")

def collect_pooled_features(rows):
    channels = expected_channels
    gap = np.empty((len(rows), channels), dtype=np.float32)
    multi = np.empty((len(rows), channels * 21), dtype=np.float32)
    for start in range(0, len(rows), BATCH_SIZE):
        stop = min(start + BATCH_SIZE, len(rows))
        maps = extract_maps(rows.resolved_path.iloc[start:stop].tolist())
        gap[start:stop] = global_average_pool(maps)
        multi[start:stop] = multi_scale_spatial_pool(maps)
    return gap, multi

train_gap, train_spatial = collect_pooled_features(train_rows)
val_gap, val_spatial = collect_pooled_features(val_rows)
y_train = train_rows.label.to_numpy()
y_val = val_rows.label.to_numpy()
print("Train/validation GAP:", train_gap.shape, val_gap.shape)
print("Train/validation multi-scale:", train_spatial.shape, val_spatial.shape)

# Report independent groups alongside correlated frame counts.
display(classification.groupby("split").agg(frames=("path", "size"), participants=("client_id", "nunique"), collection_groups=("session_id", "nunique")))


Pooled feature storage estimate: 531.1 MiB (before scaler/solver copies).


/usr/local/lib/python3.13/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input_layer_1']
Received: inputs=Tensor(shape=(8, 224, 224, 3))
  warnings.warn(msg)
/usr/local/lib/python3.13/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input_layer_1']
Received: inputs=Tensor(shape=(3, 224, 224, 3))
  warnings.warn(msg)
/usr/local/lib/python3.13/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input_layer_1']
Received: inputs=Tensor(shape=(7, 224, 224, 3))
  warnings.warn(msg)


Train/validation GAP: (2531, 2048) (559, 2048)
Train/validation multi-scale: (2531, 43008) (559, 43008)


In [ ]:
def fit_linear_probe(train_features, validation_features, train_labels, validation_labels):
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=PROBE_C, class_weight=PROBE_CLASS_WEIGHT,
                           max_iter=PROBE_MAX_ITER, solver="lbfgs", random_state=SEED),
    )
    started = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always", ConvergenceWarning)
        model.fit(train_features, train_labels)
    if any(issubclass(item.category, ConvergenceWarning) for item in caught):
        raise RuntimeError("A probe did not converge. Adjust the common iteration cap, rerun both, and document the change.")
    predictions = model.predict(validation_features)
    classifier = model[-1]
    result = {
        "feature_dimensions": int(train_features.shape[1]),
        "classifier_parameters": int(classifier.coef_.size + classifier.intercept_.size),
        "train_accuracy": float(accuracy_score(train_labels, model.predict(train_features))),
        "validation_accuracy": float(accuracy_score(validation_labels, predictions)),
        "validation_macro_f1": float(f1_score(validation_labels, predictions, average="macro", zero_division=0)),
        "validation_balanced_accuracy": float(balanced_accuracy_score(validation_labels, predictions)),
        "fit_seconds": float(time.perf_counter() - started),
    }
    return model, predictions, result

probe_models, validation_predictions, comparison_records = {}, {}, []
for name, train_features, val_features in [
    ("Global average pooling", train_gap, val_gap),
    ("Multi-scale 1×1 + 2×2 + 4×4", train_spatial, val_spatial),
]:
    model, predictions, result = fit_linear_probe(train_features, val_features, y_train, y_val)
    probe_models[name], validation_predictions[name] = model, predictions
    comparison_records.append({"representation": name, **result})

comparison = pd.DataFrame(comparison_records)
display(comparison.round(4))
delta = comparison.validation_accuracy.iloc[1] - comparison.validation_accuracy.iloc[0]
print(f"Multi-scale minus GAP validation accuracy: {100 * delta:+.2f} percentage points.")
comparison.to_csv(OUTPUT / "stage3_pooling_comparison.csv", index=False)
label_order = sorted(set(y_train))
for name, predictions in validation_predictions.items():
    display(Markdown(f"**Validation confusion matrix — {name}** (rows = actual, columns = predicted)"))
    display(pd.DataFrame(confusion_matrix(y_val, predictions, labels=label_order),
                         index=label_order, columns=label_order))


                representation  feature_dimensions  classifier_parameters  \
0       Global average pooling                2048                  14343   
1  Multi-scale 1×1 + 2×2 + 4×4               43008                 301063   

   train_accuracy  validation_accuracy  validation_macro_f1  \
0             1.0               0.7084               0.6923   
1             1.0               0.7460               0.7253   

   validation_balanced_accuracy  fit_seconds  
0                        0.7134       0.9127  
1                        0.7525      13.6260  

Multi-scale minus GAP validation accuracy: +3.76 percentage points.


**Validation confusion matrix — Global average pooling** (rows = actual, columns = predicted)

                    bent_over_row  bicep_curl  dumbbell_side_bend  push_up  \
bent_over_row                  86           1                   7        0   
bicep_curl                      7          40                  23        0   
dumbbell_side_bend              0           6                  86        0   
push_up                         0           0                   0       60   
shoulder_press                  0           0                   0        0   
squat                           9          21                   6        0   
triceps_extension               0           0                   0        0   

                    shoulder_press  squat  triceps_extension  
bent_over_row                    0      2                  0  
bicep_curl                       1      1                  0  
dumbbell_side_bend               3      1                  0  
push_up                          0     21                  0  
shoulder_press                  48      0                  0

**Validation confusion matrix — Multi-scale 1×1 + 2×2 + 4×4** (rows = actual, columns = predicted)

                    bent_over_row  bicep_curl  dumbbell_side_bend  push_up  \
bent_over_row                  91           1                   3        0   
bicep_curl                      3          45                  24        0   
dumbbell_side_bend              1           6                  85        0   
push_up                         0           0                   0       81   
shoulder_press                  0           2                   0        0   
squat                          17          34                   3        0   
triceps_extension               0           0                   0        0   

                    shoulder_press  squat  triceps_extension  
bent_over_row                    1      0                  0  
bicep_curl                       0      0                  0  
dumbbell_side_bend               4      0                  0  
push_up                          0      0                  0  
shoulder_press                  46      0                  0

### Stage 3 response — Representation tradeoff
Global average pooling achieved **70.84% validation accuracy**, while 1×1 + 2×2 + 4×4 spatial pooling achieved **74.60%**, an improvement of **3.76 percentage points** on the same 559 validation frames. The ResNet feature map has 2,048 channels: GAP produces 2,048 features, whereas the spatial representation produces 2,048 × (1 + 4 + 16) = 43,008 features. These bins retain coarse position-dependent responses, but averaging still loses fine boundaries, within-bin arrangement, person identity, and temporal information; pooling one final-layer map is not an FPN that combines multiple backbone depths.

Both probes reached 100% training accuracy on 2,531 frames, leaving substantial train/validation gaps. Macro F1 increased from 0.6923 to 0.7253 and balanced accuracy from 0.7134 to 0.7525, but the confusion matrices show uneven class effects: correct push-up predictions increased from 60/81 to 81/81, whereas correct squat predictions fell from 46/94 to 33/94. The larger probe has 301,063 classifier parameters versus 14,343 for GAP, and fitting took about 13.63 seconds versus 0.91 seconds in this run; these are probe-fitting times, not video inference speeds. The recorded feature-storage estimate was 531.1 MiB before additional scaler and solver copies.

I would retain spatial pooling as an experimental representation because of its validation improvement, while keeping GAP as the smaller baseline and investigating the squat regression before adopting it. Frames from the same participant and recording are correlated, so 559 frames are not 559 independent clients; the original participant-disjoint split reduces person leakage but does not establish generalization to new gyms, camera views, or future clients. The exact independent participant/session counts are not shown in the saved output, so I do not substitute the dataset's advertised population size. A clean rerun will print the actual group counts below; no test-set result or statistical significance is claimed here.


## Stage 4 — Design journal
The four entries below interpret the saved experiment. The course citation uses Amakobe (2026), Chapter 5; the chapter's internally labeled Section 6 ethics material appears as §5.9 in the published HTML, with “6.1 Surveillance and Consent” at §5.9.1.


### 1. Detection Architecture Justification
I recommend YOLOv8-nano as the initial detector for MIPDS trainer-reviewed video, because its single-stage design favors low latency and a small model reduces deployment compute demands (Amakobe, 2026, Chapter 5, §2.3 / published §5.5.6–5.5.8). Detection quality for small, occluded, or unusually posed people is the second deciding factor, and the missing or mislabeled push-up exercisers in this run show that it needs domain evaluation. The detector here used CPU inference on still frames; **30+ frames per second is a deployment target, not a measured result**. I would benchmark decoding, preprocessing, detection, pose extraction, and rendering together on the intended device before using live feedback. If the application becomes offline review and missing small or occluded subjects is more costly than latency, I would compare an FPN-based Faster R-CNN against YOLO using annotated domain images before changing the architecture.


### 2. Threshold Selection Rationale
I select **0.50 as a provisional trainer-review threshold**: average output fell from 5.30 boxes at 0.25 to 3.20 at 0.50, while neither setting produced zero-detection images, compared with five zero-detection images and 1.25 boxes per image at 0.75. In `8e1d58ae1ba4bd9c_00000105`, the 0.25 sports-ball label on a dumbbell disappears at 0.50, while 0.75 also removes the foreground exerciser and retains a background person. However, 0.50 loses the foreground push-up person box in `3c2d66e9f8ae3960_00000265` and retains the false horse label in `67e3fef9fbf3b516_00000188`, so it does not resolve all errors. Missed clients can prevent meaningful movement review, while extra boxes and equipment labels distract the trainer; I would expose a lower-threshold review option and require confirmation of the intended person. These scores are detection rankings, not calibrated probabilities, and the count averages do not measure recall or prove that every relevant person was found.


### 3. What the System Can Now Express
The vision module can now export a variable-length list of COCO class labels, confidence scores, and bounding boxes in original-image coordinates, rather than only a whole-image exercise label. Its separate ResNet representation can also retain coarse feature layout through 1×1, 2×2, and 4×4 pooling instead of collapsing every spatial position into one global vector. These outputs do not establish client identity, exercise phase, completed repetitions, joint geometry, or a validated coaching judgment, and the channel-mean maps are not segmentation masks. A future integration would use trainer-confirmed person selection, explicit coordinate mapping to MediaPipe landmarks, and temporal analysis for movement interpretation; those steps were not implemented or validated in this notebook.


### 4. Ethical Deployment Assessment
My primary concern is capturing and locating a nonconsenting bystander in a client's gym recording, which is concrete here because the saved detector figures include background people as well as the exerciser. Chapter 5, Section 6.1, “Surveillance and Consent,” asks whether observed people are informed, can meaningfully consent, and know who can access the data and how long it is retained (Amakobe, 2026, published §5.9.1). For personal-client use, I would request explicit recording consent, recommend a controlled camera area, and crop or redact bystanders before sharing footage or outputs. I would restrict access to the trainer and consenting client, define a retention period, and support deletion rather than retaining identifiable recordings indefinitely. This detector does not name people, but clothing, faces, mirrors, and location details can still identify them; the public dataset used here also needs attribution and does not establish permission for collecting future client recordings.


## Run record and downstream representation

Retain model provenance, input geometry, thresholds, the original split, and version information alongside results. Detector boxes are in oriented original-image pixels; the spatial summaries describe a separate resized classifier input. A downstream component should not silently equate their coordinates or treat either output as pose geometry. The example below exports the pooled features for one inspected image and matching detections only when that image was also in the 20-image detector sample.


In [ ]:
first = heatmap_images.loc[heatmap_images.scene_type.eq("single")].iloc[0]
first_map = extract_maps([first.resolved_path])
np.savez_compressed(OUTPUT / "example_scene_features.npz",
                    global_average=global_average_pool(first_map)[0],
                    multi_scale=multi_scale_spatial_pool(first_map)[0])
matching_detector_ids = detector_images.loc[detector_images.sha256.eq(first.sha256), "image_id"].tolist()
example_payload = {
    "scene_image_id": first.image_id,
    "image_sha256": first.sha256,
    "fine_tuned_backbone": BACKBONE_FAMILY,
    "spatial_map_shape_hwc": list(first_map.shape[1:]),
    "features_file": "example_scene_features.npz",
    "multi_scale_grids": [1, 2, 4],
    "feature_geometry": {"input_height_width": list(INPUT_SIZE), "resize": "stretch"},
    "detector_source": "independent COCO-pretrained YOLOv8-nano",
    "detection_results": [r for r in boxes_payload if r["image_id"] in matching_detector_ids],
    "detection_status": "included" if matching_detector_ids else "not run for this image",
    "limitations": ["no client identity", "no temporal repetition count", "no diagnostic or form judgment"],
}
(OUTPUT / "example_scene_payload.json").write_text(json.dumps(example_payload, indent=2), encoding="utf-8")

run_record = {
    "milestone": 5, "seed": SEED, "versions": versions,
    "milestone4_record_sha256": sha256_file(M4_RECORD_PATH),
    "evaluation_scope": EVALUATION_SCOPE, "classes": M4_CLASSES,
    "milestone4_task": M4_TASK, "milestone4_dataset": M4_DATASET,
    "checkpoint_sha256": sha256_file(CHECKPOINT),
    "backbone": BACKBONE_FAMILY, "nested_backbone": NESTED_BACKBONE_NAME,
    "feature_layer": FEATURE_LAYER, "input_size": list(INPUT_SIZE),
    "preprocessing": PREPROCESSING, "resize_method": RESIZE_METHOD,
    "pooling": "adaptive floor/ceil average bins; NHWC flatten; scales 1,2,4",
    "linear_probe": {"algorithm": "logistic regression", "C": PROBE_C,
                     "class_weight": PROBE_CLASS_WEIGHT, "max_iter": PROBE_MAX_ITER,
                     "standardizer_fit": "train only"},
    "split_sizes": classification.split.value_counts().to_dict(),
    "unique_clients_per_split": classification.groupby("split").client_id.nunique().to_dict(),
    "unique_sessions_per_split": classification.groupby("split").session_id.nunique().to_dict(),
    "manifest_sha256": {p.name: sha256_file(p) for p in schemas},
    "detector": {"weights": YOLO_WEIGHTS, "confidence_thresholds": list(THRESHOLDS),
                 "iou": YOLO_IOU, "imgsz": YOLO_IMGSZ, "device": str(YOLO_DEVICE)},
    "claim_boundary": "validation representation comparison and qualitative 20-image detection analysis",
}
(OUTPUT / "run_record.json").write_text(json.dumps(run_record, indent=2), encoding="utf-8")
print("Saved the real run's figures, tables, detector boxes, example features, and provenance.")


/usr/local/lib/python3.13/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['input_layer_1']
Received: inputs=Tensor(shape=(1, 224, 224, 3))
  warnings.warn(msg)


Saved the real run's figures, tables, detector boxes, example features, and provenance.



### Original Colab runtime report (before written completion)
The following saved report predates the written edits: its qualitative-response check was False because the dictionary was empty. Those assessments are now filled, and the summary above has been recomputed from the saved per-image results. The five original computational checks passed; this editing pass did not access the Drive files or rerun the full Colab pipeline. Rerun the summary and final checks in Colab to update the runtime report and exported summary CSV.


In [ ]:
checks = {
    "Three heatmap figure files": len(heatmap_files) == 3 and all(p.is_file() for p in heatmap_files),
    "20 images at each of 3 thresholds": len(detections) == 60 and detections.groupby("threshold").size().eq(20).all(),
    "Qualitative sentences entered": all(QUALITATIVE_ASSESSMENT[t].strip() for t in THRESHOLDS),
    "Both validation accuracies computed": len(comparison) == 2 and comparison.validation_accuracy.notna().all(),
    "Correct multi-scale dimension": spatial_example.shape[1] == expected_channels * 21,
    "Run provenance exported": (OUTPUT / "run_record.json").is_file(),
}
display(pd.DataFrame([{"check": key, "passed": bool(value)} for key, value in checks.items()]))
print("Written responses and Chapter 5 Section 6 citation are completed; review them alongside the saved figures.")
print("For final reproducibility, run all cells in a clean Colab runtime and resolve any errors; preserve the original outputs first.")
print("In Colab: File → Download → Download .ipynb; retain rendered outputs in the submitted notebook.")


                                 check  passed
0           Three heatmap figure files    True
1    20 images at each of 3 thresholds    True
2        Qualitative sentences entered   False
3  Both validation accuracies computed    True
4        Correct multi-scale dimension    True
5              Run provenance exported    True

Manual review still required: replace every response/journal placeholder and cite Chapter 5 Section 6.
After edits, restart the runtime and run all cells. Resolve every error before downloading.
In Colab: File → Download → Download .ipynb; retain rendered outputs in the submitted notebook.


## Post-edit review and submission checklist

The written responses have been completed from the actual saved results, all 60 detector figure panels have been inspected, and the threshold summary was refreshed without changing any inference result. The original computational outputs and pre-edit runtime report remain visible. Code cells were checked for Python syntax; a full model/data run was not repeated during this editing pass.

- [x] Actual Milestone 4 checkpoint and original classification manifest passed the saved hash checks.
- [x] Three heatmaps are present in the original reviewed export, with the three interpretations in this copy.
- [x] Twenty images at each of three thresholds and single-sentence assessments are present.
- [x] Both probe validation accuracies and confusion matrices are retained.
- [x] All four design-journal entries contain 3–5 sentences and the ethics entry cites the assigned chapter.
- [x] Scene descriptions acknowledge the first image's background people and correct the third image's inaccurate “holding a dumbbell” description.
- [ ] A final clean Colab rerun after these edits, including the displayed independent-group counts and updated exported summary, remains to be verified.
- [ ] Confirm the specific milestone rubric accepts a dominant centered foreground subject with background people and a mixed person/equipment scene; the source rubric is not embedded in this upload.
- [ ] Apply any required course AI-use disclosure and confirm permissions for submitted visual material.

### Course reference

Amakobe, M. (2026). *Deep learning: A comprehensive guide: Architectures, training, generative models, and real-world AI systems* (First Open Edition). https://proff-amakobe.github.io/oer-books/deep-learning/

Assigned chapter: [Chapter 5 — Visual Tasks Beyond Classification](https://proff-amakobe.github.io/oer-books/deep-learning/chapters/05-visual-tasks-beyond-classification.html); Section 6, “Ethics in the Pixel,” is published as §5.9, including [Section 6.1 — Surveillance and Consent](https://proff-amakobe.github.io/oer-books/deep-learning/chapters/05-visual-tasks-beyond-classification.html#surveillance-and-consent). The citation follows the author's published APA recommendation.

### Implementation references

- [Keras ResNet preprocessing and spatial outputs](https://keras.io/api/applications/resnet/resnet_models/)
- [Keras inference-mode feature extraction](https://keras.io/guides/transfer_learning/)
- [Ultralytics YOLOv8](https://docs.ultralytics.com/models/yolov8/)
- [Ultralytics prediction settings and result boxes](https://docs.ultralytics.com/modes/predict/)
- [Adaptive average pooling convention](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.adaptive_avg_pool2d.html)
- [scikit-learn LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)
- [Public exercise dataset — Mendeley Data V3, CC BY 4.0](https://data.mendeley.com/datasets/kgbb3yn47p/3)
